In [41]:
import pandas as pd
import numpy as np
import matplotlib as plt

In [42]:
filepath = r"C:\Users\Lenovo\Documents\GitHub\Coffee_Reco_System\data\raw\coffee_reviewsV2.csv"
kaggle_filepath= r"C:\Users\Lenovo\Documents\GitHub\Coffee_Reco_System\data\raw\kaggle_coffee_analysis.csv"

In [43]:
df = pd.read_csv(filepath)

In [44]:
df.head(5)

,url,coffee_name,score,review,roast_level,review_date
0,https://www.coffeereview.com/review/ecuador-la...,Ecuador La Papaya Pacamara,95,"Intricate, richly sweet-savory. Apple cider, s...",Light,Apr-25
1,https://www.coffeereview.com/review/ka%ca%bbu-...,Kaʻū Yellow Bourbon,95,"Richly floral, high-toned. Wisteria, dried pea...",Medium-Light,May-25
2,https://www.coffeereview.com/review/laos-xam-tai/,Laos Xam Tai,92,"Delicately sweet, gently herbaceous. Cocoa nib...",Light,Apr-25
3,https://www.coffeereview.com/review/colombia-s...,Colombia San Adolfo Huila,95,"Delicate, tropical, cocoa-toned. Cocoa nib, pa...",Light,Apr-25
4,https://www.coffeereview.com/review/kenya-kiri...,Kenya Kirinyaga Karamundi Washed AB,95,"Richly sweet-savory. Ripe tomato, red currant,...",Light,May-25


In [45]:
counts = df.groupby('coffee_name')['review'].size()
print("Max reviews: ",counts.max())
print("Max reviews: ",counts.min())
print("Coffees with only 1 review: ", (counts == 1).sum())

Max reviews:  30
Max reviews:  1
Coffees with only 1 review:  6418


In [46]:
df.groupby('coffee_name').size().describe()

count    7030.000000
mean        1.185349
std         1.052572
min         1.000000
25%         1.000000
50%         1.000000
75%         1.000000
max        30.000000
dtype: float64

In [47]:
reviews = df.groupby('coffee_name').size()
print("Coffee with 1 review:", (reviews == 1).sum())
print("Coffee with 3 reviews:", (reviews >= 3).sum())
print("Coffee with over 5 reviews:", (reviews >= 5).sum())

Coffee with 1 review: 6418
Coffee with 3 reviews: 223
Coffee with over 5 reviews: 75


In [48]:
print(df.shape)
print(df['coffee_name'].nunique())

(8333, 6)
7030


In [49]:
kaggle_df = pd.read_csv(r"C:\Users\Lenovo\Documents\GitHub\Coffee_Reco_System\data\raw\kaggle_coffee_analysis.csv")

In [50]:
overlap = set(df['coffee_name'].str.lower().str.strip()) & set(kaggle_df['name'].str.lower().str.strip())
print(len(overlap))

1906


In [51]:
kaggle_df = pd.read_csv(kaggle_filepath)
print(kaggle_df.columns.tolist())
print(kaggle_df.head(2))
kaggle_df.shape[0]

['name', 'roaster', 'roast', 'loc_country', 'origin_1', 'origin_2', '100g_USD', 'rating', 'review_date', 'desc_1', 'desc_2', 'desc_3']
                      name roaster         roast loc_country origin_1  \
0  “Sweety” Espresso Blend  A.R.C.  Medium-Light   Hong Kong   Panama   
1     Flora Blend Espresso  A.R.C.  Medium-Light   Hong Kong   Africa   

       origin_2  100g_USD  rating    review_date  \
0      Ethiopia     14.32      95  November 2017   
1  Asia Pacific      9.05      94  November 2017   

                                              desc_1  \
0  Evaluated as espresso. Sweet-toned, deeply ric...   
1  Evaluated as espresso. Sweetly tart, floral-to...   

                                              desc_2  \
0  An espresso blend comprised of coffees from Pa...   
1  An espresso blend comprised of coffees from Af...   

                                              desc_3  
0  A radiant espresso blend that shines equally i...  
1  A floral-driven straight shot, amplif

2095

In [56]:
import pandas as pd

def load_data(filepath: str) -> pd.DataFrame:
    df = pd.read_csv(filepath)
    return df

def validate_schema(df: pd.DataFrame) -> pd.DataFrame:
    required_cols = ['coffee_name', 'score', 'review', 'roast_level']
    assert not df.empty, "Dataset is empty"
    assert all(col in df.columns for col in required_cols), \
        f"Missing columns! Required: {required_cols}, Found: {list(df.columns)}"
    return df

def prepare_kaggle(kaggle_filepath: str) -> pd.DataFrame:
    kaggle_df = pd.read_csv(kaggle_filepath)

    kaggle_df = pd.melt(
                kaggle_df,
                id_vars=["name", "roast", "rating"],  
                value_vars=["desc_1", "desc_2", "desc_3"],          
                value_name="review"                                 
                )

    kaggle_df = kaggle_df.rename(columns={
        "name": "coffee_name",
        "roast": "roast_level",
        "rating": "score"
    })

    kaggle_df = kaggle_df.drop(columns=["variable"])

    kaggle_df = kaggle_df[kaggle_df["review"].notna() & (kaggle_df["review"].str.strip() != "")]

    kaggle_df = kaggle_df[["coffee_name", "roast_level", "score", "review"]]

    return kaggle_df
    
def load_and_merge_kaggle(df: pd.DataFrame, kaggle_filepath: str) -> pd.DataFrame:
    kaggle_df = pd.read_csv(kaggle_filepath)
    
    kaggle_df = kaggle_df[["coffee_name", "review", "roast_level"]]
    merged_df = pd.concat([df, kaggle_df], ignore_index=True)
    
    return merged_df

def filter_by_review_density(df: pd.DataFrame, min_reviews: int = 3) -> pd.DataFrame:
    review_count = df.groupby('coffee_name')['review'].size()
    coffee_with_many_reviews = review_count[review_count >= min_reviews].index

    filtered_df = df[df["coffee_name"].isin(coffee_with_many_reviews)]
    
    return filtered_df

In [57]:
def normalize_name(df: pd.DataFrame) -> pd.DataFrame:
    df["coffee_name"] = (
        df["coffee_name"]
        .str.strip()
        .str.lower()
        .str.replace(r"[^a-z0-9\s]", "", regex=True)
        .str.replace(r"\s+", " ", regex=True)  # collapse multiple spaces into one
    )
    return df

In [58]:
def build_pipeline(filepath: str, kaggle_filepath: str) -> pd.DataFrame:
    df = load_data(filepath)
    
    df = validate_schema(df)
    
    df = normalize_name(df)
    
    kaggle_df = prepare_kaggle(kaggle_filepath)
    
    kaggle_df = normalize_name(kaggle_df)
    
    overlap_mask = kaggle_df["coffee_name"].isin(df["coffee_name"])
    kaggle_overlap = kaggle_df[overlap_mask]
    merged_df = pd.concat([df, kaggle_overlap], ignore_index=True)

    filtered_df = filter_by_review_density(merged_df)

    return filtered_df

In [59]:
final_df = build_pipeline(filepath, kaggle_filepath)
print(final_df.shape)
print(final_df["coffee_name"].nunique())
print(final_df.groupby("coffee_name").size().describe())

(9358, 6)
2016
count    2016.000000
mean        4.641865
std         2.639406
min         3.000000
25%         4.000000
50%         4.000000
75%         4.000000
max        46.000000
dtype: float64


In [31]:
kaggle_prepared = prepare_kaggle(kaggle_filepath)
print(kaggle_prepared.shape)
print(kaggle_prepared["review"].isna().sum())  # remaining nulls
print(kaggle_prepared.groupby("coffee_name").size().describe())  # review distribution

(6283, 4)
0
count    1909.000000
mean        3.291252
std         1.236682
min         2.000000
25%         3.000000
50%         3.000000
75%         3.000000
max        18.000000
dtype: float64
